# Load data

In [1]:
import sys
sys.path.append('../../Common/')

import CommonYFinance, CommonBinance, CommonMT5, CommonBacktest

In [2]:
def detectSignal(symbol, from_date, to_date, timeframe):

	import pandas as pd
	import plotly.graph_objects as go
	import redis
	import numpy as np
	import talib
	from datetime import datetime

	# ##############################################Step 1: Lấy dữ liệu##############################################
	data = CommonBinance.CommonBinance.loaddataBinance_FromTo(symbol, from_date, to_date, timeframe)
	
	# ##############################################Step 2: Chien luoc##############################################  
	# Thiết lập cửa sổ thời gian cho SMA và độ lệch chuẩn
	window = 20
	# Tính toán SMA cho giá đóng cửa
	data['SMA'] = talib.SMA(data['Close'], timeperiod=window)

	# Tính toán MACD
	data['MACD'], data['Signal_Line'], data['Hist'] = talib.MACD(data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)

	# Tạo cột tín hiệu mua/bán
	# Mot tin hieu mua xay ra khi: MACD > Signal_Line va gia dong cua > SMA
	# Mot tin hieu ban xay ra khi: MACD < Signal_Line va gia dong cua < SMA
	data['Buy_Signal'] = (data['MACD'] > data['Signal_Line']) & (data['Close'] > data['SMA'])
	data['Sell_Signal'] = (data['MACD'] < data['Signal_Line']) & (data['Close'] < data['SMA'])    

	# ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
	# Nếu có tín hiệu thì đẩy qua Redis, con không thì không làm gì cả, chi in ra man hinh

	# Tạo kết nối Redis
	r = redis.Redis(host='localhost', port=6379, db=15) # 0-5: CK, 6-10: Crypto, 11-15: Forex
	# Tạo hash key
	hash_key = 'OG_CRTO_MACD,MA_K12_TEST'
	last_record = data.iloc[-1]
	
	# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
	if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
		for field, value in last_record.to_dict().items():
			# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
			if isinstance(value, pd.Timestamp):
				value = value.isoformat()
			elif isinstance(value, (int, np.uint64)):
				value = str(value)
			r.hset(hash_key, field, value)  
			r.hset(hash_key, 'Symbol', symbol)
			r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
		print(last_record)   
	else:
		print('Không có tín hiệu!')
	# ##############################################Step 4: Vẽ biểu đồ##############################################  
	# data.set_index('Datetime', inplace=True)
	# # Tạo figure
	# fig = go.Figure()
	# # Biểu đồ giá đóng cửa và SMA
	# fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Giá Đóng Cửa', line=dict(color='blue')))
	# fig.add_trace(go.Scatter(x=data.index, y=data['SMA'], mode='lines', name='SMA', line=dict(color='orange')))
	# # Vẽ các điểm cho tín hiệu mua
	# buy_signals = data[data['Buy_Signal']]
	# fig.add_trace(go.Scatter(x=buy_signals.index, y=buy_signals['Close'], mode='markers', name='Buy Signal', marker=dict(symbol='triangle-up', size=10, color='green')))
	# # Vẽ các điểm cho tín hiệu bán
	# sell_signals = data[data['Sell_Signal']]
	# fig.add_trace(go.Scatter(x=sell_signals.index, y=sell_signals['Close'], mode='markers', name='Sell Signal', marker=dict(symbol='triangle-down', size=10, color='red')))
	# fig.update_layout(title='Giá Đóng Cửa và SMA', yaxis_title='Giá', xaxis_rangeslider_visible=False, height=500)
	# # Hiển thị biểu đồ
	# fig.show()

	# ####################################################################################################
	# # Tạo figure
	# fig = go.Figure()
	# # Biểu đồ MACD và Signal Line
	# fig.add_trace(go.Scatter(x=data.index, y=data['MACD'], mode='lines', name='MACD', line=dict(color='blue')))
	# fig.add_trace(go.Scatter(x=data.index, y=data['Signal_Line'], mode='lines', name='Signal Line', line=dict(color='orange')))
	# fig.update_layout(title='MACD và Signal Line', yaxis_title='MACD', xaxis_rangeslider_visible=False, height=500)
	# # Hiển thị biểu đồ
	# fig.show()

In [ ]:
from datetime import datetime, timedelta
import schedule
import time

def schedule_detectSignal():
	# Truyen cac tham so khoi tao cho ham detectSignal
	symbol = 'NEARUSDT'
	from_date = (datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
	to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
	timeframe = '1m' # 1m
	detectSignal(symbol, from_date, to_date, timeframe)

# Danh sách các phút cụ thể bạn muốn hàm được chạy
# Khai bao danh sách các phút cụ thể mà bạn muốn hàm được chạy
# Ví dụ: chạy hàm moi phut chay 1 lan
run_minutes = list(range(0, 60, 1))  # Tất cả các phút từ 0 đến 59: 0, 1, 2, ..., 59
# Hoặc nếu bạn chỉ muốn chạy hàm vào các phút cụ thể, bạn có thể thay đổi danh sách này

# Set Schedule o day
# Chien luoc cua chung ta: 1 phut danh 1 lan
while True:
	# Lấy thời gian hiện tại
	current_time = datetime.now() # Lấy thời gian hiện tại cua may tinh
	current_minute = current_time.minute # Lấy phút hiện tại

	# Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
	if current_minute in run_minutes:
		# Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
		last_run = getattr(schedule_detectSignal, 'last_run', None) # Ham getatattr() sẽ trả về giá trị của thuộc tính 'last_run' nếu nó tồn tại, nếu không thì trả về None
		# Nếu chưa chạy hoặc đã chạy ở phút khác thì gọi hàm
		if last_run is None or last_run != current_minute:
			schedule_detectSignal()
			# Lưu lại phút cuối cùng mà hàm đã chạy
			setattr(schedule_detectSignal, 'last_run', current_minute)   

	# Nghỉ 10 giây trước khi kiểm tra lại
	# time.sleep(10)


Datetime       2025-07-24 15:24:00
Open                         2.807
High                         2.807
Low                          2.804
Close                        2.804
Volume                      4011.6
SMA                        2.81575
MACD                       0.00571
Signal_Line               0.009654
Hist                     -0.003945
Buy_Signal                   False
Sell_Signal                   True
Name: 924, dtype: object
Datetime       2025-07-24 15:25:00
Open                         2.804
High                         2.804
Low                          2.803
Close                        2.804
Volume                     12210.2
SMA                         2.8159
MACD                      0.004539
Signal_Line               0.008631
Hist                     -0.004092
Buy_Signal                   False
Sell_Signal                   True
Name: 925, dtype: object
Datetime       2025-07-24 15:26:00
Open                         2.804
High                         2.805
Low  